# Meta-Learning (Stacking) Ensemble
Neste notebook implementamos a técnica de Stacking (Meta-learning), treinando um modelo secundário (XGBoost, Regressão Logística, MLP, etc.) sobre as predições (probabilidades ou logits) dos modelos base.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## 1. Carregar Predições dos Modelos Base
Para treinar o meta-modelo de forma correta sem data leakage, precisamos usar as predições Out-of-Fold (OOF) do conjunto de treino ou usar um conjunto de validação *hold-out* que os modelos base não viram.

In [ ]:
# Exemplo: Carregando CSVs com as predições (Ajuste os caminhos para as suas predições OOF e de Teste reais)
# df_oof_b0 = pd.read_csv('preds_oof/b0_oof.csv')
# df_oof_v2s = pd.read_csv('preds_oof/v2s_oof.csv')

# df_test_b0 = pd.read_csv('preds_test/b0_test.csv')
# df_test_v2s = pd.read_csv('preds_test/v2s_test.csv')

print("Carregando as predições dos modelos base...")

## 2. Preparar Features (X) e Target (y)
As features do meta-modelo serão as probabilidades (ou logits) das classes previstas pelos modelos base combinadas.

In [ ]:
# Supondo que df_oof_b0[['prob_0', 'prob_1', ...]] contenha as predições:

# X_meta_train = np.hstack([
#     df_oof_b0[[f'prob_{i}' for i in range(6)]].values,
#     df_oof_v2s[[f'prob_{i}' for i in range(6)]].values
# ])
# y_meta_train = df_oof_b0['target'].values

# X_meta_test = np.hstack([
#     df_test_b0[[f'prob_{i}' for i in range(6)]].values,
#     df_test_v2s[[f'prob_{i}' for i in range(6)]].values
# ])
# y_meta_test = df_test_b0['target'].values

print("Features prontas: X_meta_train, y_meta_train, X_meta_test")

## 3. Treinar Meta-Modelos
Vamos experimentar diferentes modelos: Regressão Logística, XGBoost e LightGBM.

In [ ]:
def evaluate_meta_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    print(f"[{name}] Accuracy: {acc:.4f} | QWK: {kappa:.4f}")
    return kappa

In [ ]:
# ==========================================
# 3.1. Regressão Logística
# ==========================================
# A regressão logística é um excelente meta-modelo porque tende a não sofrer overfitting facialmente.

# lr_meta = LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42)
# lr_meta.fit(X_meta_train, y_meta_train)
# lr_preds = lr_meta.predict(X_meta_test)

# lr_kappa = evaluate_meta_model("Regressão Logística", y_meta_test, lr_preds)

In [ ]:
# ==========================================
# 3.2. XGBoost
# ==========================================
# Modelos baseados em árvore podem descobrir não-linearidades entre as predições dos modelos base.

# xgb_meta = xgb.XGBClassifier(
#     eval_metric='mlogloss',
#     max_depth=3,
#     learning_rate=0.05,
#     n_estimators=100,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42
# )
# xgb_meta.fit(X_meta_train, y_meta_train)
# xgb_preds = xgb_meta.predict(X_meta_test)

# xgb_kappa = evaluate_meta_model("XGBoost", y_meta_test, xgb_preds)

In [ ]:
# ==========================================
# 3.3. LightGBM
# ==========================================

# lgb_meta = lgb.LGBMClassifier(
#     max_depth=3,
#     learning_rate=0.05,
#     n_estimators=100,
#     random_state=42
# )
# lgb_meta.fit(X_meta_train, y_meta_train)
# lgb_preds = lgb_meta.predict(X_meta_test)

# lgb_kappa = evaluate_meta_model("LightGBM", y_meta_test, lgb_preds)

## 4. Otimização do Meta-Modelo usando K-Fold (Para evitar overfitting)
Podemos usar Validação Cruzada (StratifiedKFold) no X_meta_train para ter uma estimativa de QWK OOF mais robusta para o meta-modelo.

In [ ]:
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# oof_meta_preds = np.zeros(len(X_meta_train))
# test_meta_preds = np.zeros((len(X_meta_test), 6))

# for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta_train, y_meta_train)):
#     X_tr, y_tr = X_meta_train[train_idx], y_meta_train[train_idx]
#     X_va, y_va = X_meta_train[val_idx], y_meta_train[val_idx]
    
#     model = LogisticRegression(max_iter=1000, multi_class='multinomial')
#     model.fit(X_tr, y_tr)
    
#     # Predições no Fold de Validação
#     oof_meta_preds[val_idx] = model.predict(X_va)
    
#     # Predições no Teste (Probabilidades)
#     test_meta_preds += model.predict_proba(X_meta_test) / skf.n_splits

# final_test_preds = np.argmax(test_meta_preds, axis=1)
# print("OOF QWK Meta-Model:", cohen_kappa_score(y_meta_train, oof_meta_preds, weights='quadratic'))
# evaluate_meta_model("K-Fold Meta-Model", y_meta_test, final_test_preds)